## Analyse descriptive
# Introduction

Ce rapport présente une analyse descriptive complète des données horaires de qualité de l'air (concentration en PM2.5) collectées en Ile-de-France sur la période 2021-2025. L'objectif est d'identifier les tendances, la saisonnalité et les variations spatiales de la pollution aux particules fines.

Les données proviennent de 18 stations de mesure réparties en Ile-de-France, avec un focus particulier sur les zones urbaines, industrielles et rurales. Les variables analysées incluent la concentration en PM2.5, les conditions météorologiques (température, humidité, vitesse du vent, précipitations) et la densité d'installations industrielles.


In [ ]:
import sys
import os
import pandas as pd
sys.path.insert(0, os.path.abspath(".."))

## Importation de la base de données

In [ ]:
df = pd.read_csv("https://minio.lab.sspcloud.fr/ganlea/projet-qualite-air/processed/dataset_consolide.csv")
df["datetime_debut"]=pd.to_datetime(df["datetime_debut"])

## 1. ANALYSE DESCRITPIVE DE LA VARIABLE CIBLE (PM2.5)

On commence par une visualisation des types des variables contenues dans la base de données

In [ ]:
print(df.dtypes)

Une visualisation d'ensemble de chaque variable quantitative permet d'avoir une idée sur la distribution et le comportement de ces variables.

In [ ]:
liste_variables_quanti = ["pm25_brute", "vent_vitesse_ms", "vent_direction_deg", "temperature_c", "humidite_pct",
                            "pluie_mm", "nb_installations_5km"]
variables_quantitatives = df[liste_variables_quanti]
variables_quantitatives.describe(include="all")

PM2.5 (Particules fines) : La moyenne observée est de 10.15 µg/m³, bien en dessous du seuil réglementaire de 25 µg/m³. Cependant, l'écart-type de 7.93 indique une forte variabilité. Le maximum atteint 190.33 µg/m³, suggérant des pics de pollution importants. La médiane (8.00) étant inférieure à la moyenne, la distribution est asymétrique vers la droite.

Vent (Vitesse) : Vitesse moyenne de 3.04 m/s, relativement modérée. L'écart-type de 1.82 m/s indique une variabilité normale. Les vitesses varient de 0 m/s (calme plat) à 17.8 m/s (venteux).

Température : La température moyenne observée est de 13°C. L'écart-type de 7.2°C suggère des variations saisonnières modérées. Les valeurs négatives (-10.7°C) et positives (40.4°C) indiquent la présence de périodes hivernales et estivales.

Humidité : Humidité moyenne de 75.18%, plutôt élevée. La distribution est concentration entre 63% (Q1) et 90% (Q3), suggérant un climat humide.

Pluie : Précipitations moyennes très faibles (0.08 mm/heure), avec 75% des observations sans pluie. Cela reflète le fait que les précipitations sont des événements localisés dans le temps.

Industries (densité dans rayon 5km) : Moyenne de 5.15 installations par station, avec forte variabilité (6.38). Les zones rurales peuvent avoir 0 industrie, tandis que certains sites près de zones industrielles en comptent jusqu'à 22. C'est une proxy de la pollution locale d'origine structurelle.

On vérifiera dans la suite la pertinence de ces variables dans notre analyse.


### Diagramme en barre des dépassements de seuil de concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_frequence_depassement

print(graphique_frequence_depassement(df, "depasse_seuil_24h"))

Ce graphique illustre la distribution binaire de la conformité réglementaire des concentrations en PM2.5 en Île-de-France sur la période 2021-2025, basée sur le seuil européen de 25 µg/m³. L'analyse révèle une asymétrie marquée : 81.3% des observations restent en dessous du seuil, tandis que 18.7%  le dépassent. Cette distribution montre que, bien que la qualité de l'air générale soit acceptable, des épisodes de pollution significatifs surviennent régulièrement. Le taux de dépassement de 18.7% équivaut à environ 1 jour sur 5.4 où les concentrations excèdent la limite réglementaire.


### Diagramme en barre des stations avec les mesures de concentrations moyennes les plus hautes et les plus basses


In [ ]:
from src.data_preprocessing import graphique_top_bottom_stations

print(graphique_top_bottom_stations(df, n=5, colonne="pm25_brute", station="nom_station"))

Ce graphique en barres horizontales compare les concentrations moyennes de PM2.5 entre les 5 stations les plus polluées (en rouge) et les 5 moins polluées (en vert) en Île-de-France. Les résultats révèlent ce qui suit : Auto A1 Saint-Denis enregistre 14.16 µg/m³ (2.16× supérieur à Zone Rurale SE avec 6.56 µg/m³), un écart de 7.60 µg/m³. Les stations routières dominent le classement des plus polluées (Auto A1, RN20, Bld Périphérique à 13.04-14.16 µg/m³), reflétant l'impact direct du trafic automobile intensif et des émissions d'échappement. À l'inverse, les zones rurales (Zones Rurale SE/Sud/Nord à 6.56-8.05 µg/m³) bénéficient d'une meilleure ventilation atmosphérique et d'une distance aux sources d'émission. Les stations urbaines parisiennes (Paris 1er, Boulevard Haussmann à 9.58-11.23 µg/m³) occupent une position intermédiaire.

### Distribution de concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_histogramme_pm25

print(graphique_histogramme_pm25(df, colonne="pm25_brute"))

Ce graphique superpose l'histogramme de fréquence (barres bleues) et la courbe de densité KDE (ligne rouge) des concentrations en PM2.5, révélant une distribution fortement asymétrique vers la droite. Le pic principal concentre les observations autour de 8 µg/m³ (la médiane), confirmant que la majorité des mesures reste bien en dessous du seuil alerte (ligne pointillée jaune à 25 µg/m³). Cependant, la queue de distribution s'étend jusqu'à environ 200 µg/m³, indiquant des valeurs extrêmes rares mais significatives. Cette forme est caractéristique des données de pollution environnementale où les conditions normales dominent mais des événements exceptionnels (inversions thermiques, épisodes transfrontaliers) génèrent des pics isolés.

### Boxplot de la concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_boxplot_global

print(graphique_boxplot_global(df, colonne="pm25_brute"))

Ce boxplot synthétise la distribution complète des PM2.5 à travers une représentation robuste des quartiles et des valeurs extrêmes. Le seuil alerte (25 µg/m³, ligne pointillée rouge) dépasse largement la boîte, révélant que les dépassements nécessitent une accumulation significative de polluants au-delà des conditions normales. Les points noirs au-dessus de la boîte représentent des valeurs extrêmes. Ces observations confirment les analyses faites jusqu'à présent.

## 2. ANALYSE TEMPORELLE DE LA VARIABLE CIBLE (PM2.5)

In [ ]:
from src.data_preprocessing import tableau_moyennes_mensuelles, graphique_moyennes_mensuelles

print(tableau_moyennes_mensuelles(df, "pm25_brute", "mois"))
print(graphique_moyennes_mensuelles(df, "pm25_brute", "mois"))

In [ ]:
from src.data_preprocessing import graphique_moyennes_saisonnieres

print(graphique_moyennes_saisonnieres(df, "pm25_brute", "saison"))

In [ ]:
from src.data_preprocessing import graphique_boxplot_mensuel

print(graphique_boxplot_mensuel(df, "pm25_brute", "mois"))

In [ ]:
from src.data_preprocessing import graphique_serie_temporelle

print(graphique_serie_temporelle(df, "pm25_brute", "datetime_debut"))

In [ ]:
from src.data_preprocessing import tableau_moyennes_jour_semaine

print(tableau_moyennes_jour_semaine(df, "pm25_brute", "jour_semaine"))

In [ ]:
from src.data_preprocessing import graphique_boxplot_jour_semaine

print(graphique_boxplot_jour_semaine(df, "pm25_brute", "jour_semaine"))

In [ ]:
from src.data_preprocessing import graphique_cycle_diurne

print(graphique_cycle_diurne(df, "pm25_brute", "heure"))

In [ ]:
from src.data_preprocessing import graphique_heatmap_heure_jour

print(graphique_heatmap_heure_jour(df, "pm25_brute", "heure", "jour_semaine"))

## 3. ANALYSE SPATIALE DE LA VARIABLE CIBLE (PM2.5)

In [ ]:
from src.data_preprocessing import tableau_comparaison_stations

print(tableau_comparaison_stations(df, "nom_station", "pm25_brute", "depasse_seuil_24h", "datetime_debut", "nb_installations_5km"))

In [ ]:
from src.data_preprocessing import graphique_boxplot_stations

print(graphique_boxplot_stations(df, "nom_station", "pm25_brute"))

In [ ]:
from src.data_preprocessing import graphique_bar_depassements_stations

print(graphique_bar_depassements_stations(df, "nom_station", "depasse_seuil_24h"))

## 4. ANALYSE DES CORRÉLATIONS (VARIABLES MÉTÉO ET D'INFRASTRUCTURES) AVEC LA VARIABLE CIBLE PM2.5

In [ ]:
from src.data_preprocessing import graphique_heatmap_correlation
list_variable = ["pm25_brute", "vent_vitesse_ms", "vent_direction_deg", "temperature_c", "humidite_pct", "pluie_mm"]
print(graphique_heatmap_correlation(df, list_variable))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "vent_vitesse_ms", "pm25_brute", "Vitesse du vent", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "vent_vitesse_ms"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "vent_direction_deg", "pm25_brute", "Direction du vent", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "vent_direction_deg"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "temperature_c", "pm25_brute", "Température", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "temperature_c"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "humidite_pct", "pm25_brute", "Humidité", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "humidite_pct"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "pluie_mm", "pm25_brute", "Précipitations mm", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import tracer_comparaison_carto

print(tracer_comparaison_carto(df, 2024, "nom_station", "pm25_brute", "nb_installations_5km", "lat", "lon", "annee"))

## 5. ANALYSE DE PERSISTANCE (AUTO-CORRÉLATION)

In [ ]:
from src.data_preprocessing import graphique_acf_pm25

print(graphique_acf_pm25(df, "pm25_brute", "datetime_debut"))

In [ ]:
from src.data_preprocessing import graphique_pacf_pm25

print(graphique_pacf_pm25(df, "pm25_brute", "datetime_debut"))